In [1]:
import pathlib
import sys

_here = pathlib.Path.cwd().resolve()
for _parent in [_here, *_here.parents]:
    if (_parent / "src" / "quant_textbook").exists():
        sys.path.insert(0, str(_parent / "src"))
        break

# 42. B8 — Bayesian inference and latent-state uncertainty

> B8はB7とは別のmarket storyを作らない。同じTreasury curve、同じ5公表日先target、同じouter testで、parameter・predictive・state uncertaintyを追加する。

## 学習目標

- prior、likelihood、posterior、posterior predictiveを分けられる
- partial poolingとno/complete poolingを比較できる
- DAG、mixture、HMMのconditional independenceを読める
- MCMCをacceptanceだけでなくESSとmulti-chain diagnosticで監査できる
- latent stateを観測された「真のregime」と呼ばず予測分布として評価できる

## 前提知識

- B2の条件付き確率とMarkov chain
- B3のlikelihoodとfinite-sample uncertainty
- B7のfactor dynamics、filtered/smoothed distinction

In [2]:
import time

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

import quant_textbook as qt

pio.renderers.default = "notebook_connected"
RANDOM_SEED = 20260810
NOTEBOOK_ID = 42


def task_rng(task_id, *coordinates):
    entropy = [
        RANDOM_SEED,
        NOTEBOOK_ID,
        int(task_id),
        *(int(coordinate) for coordinate in coordinates),
    ]
    return np.random.default_rng(np.random.SeedSequence(entropy))

In [3]:
treasury = qt.load_treasury_snapshot()
rates = treasury.frame.copy()
forecast = qt.make_treasury_forecast_dataset(rates)
b5_split = qt.chronological_split(len(forecast.regression_target), gap=1)

maturity_years = np.array([0.25, 2.0, 5.0, 10.0, 30.0])
curve_yields = rates.loc[:, qt.DEFAULT_TENORS].to_numpy(dtype=float)
curve_dates = rates["date"].to_numpy(dtype="datetime64[ns]")
curve_changes_bp = np.diff(curve_yields, axis=0) * 100.0
change_dates = curve_dates[1:]

train_end_date = forecast.prediction_dates[b5_split.train.max()]
validation_end_date = forecast.prediction_dates[b5_split.validation.max()]
test_start_date = forecast.prediction_dates[b5_split.test.min()]
train_mask = curve_dates <= train_end_date
validation_mask = (curve_dates > train_end_date) & (curve_dates <= validation_end_date)
test_mask = curve_dates >= test_start_date

assert treasury.quality.accepted
assert train_end_date < validation_end_date < test_start_date
assert np.all(np.isfinite(curve_yields))
assert np.all(np.diff(curve_dates).astype("timedelta64[D]") > np.timedelta64(0, "D"))

print("source:", treasury.metadata.source_name)
print("snapshot:", treasury.metadata.start_date, "to", treasury.metadata.end_date)
print("curve rows / tenors:", curve_yields.shape)
print("B5 train / validation end:", train_end_date, validation_end_date)
print("locked outer-test start:", test_start_date)
print("snapshot sha256:", treasury.metadata.snapshot_sha256)

source: U.S. Treasury Daily Par Yield Curve Rates
snapshot: 2015-01-02 to 2025-12-31
curve rows / tenors: (2750, 5)
B5 train / validation end: 2021-08-11T00:00:00.000000000 2023-10-19T00:00:00.000000000
locked outer-test start: 2023-10-23T00:00:00.000000000
snapshot sha256: 6ddef9605abbf02c6a4526a51f098135b41da1a437915623af672b1c7bcbd295


## 1. B8 evidence chain

| Week | Core | Treasury lab | 主な反証 |
|---|---|---|---|
| 29 | conjugacy、prior/posterior predictive | 5日先10年変化 | prior sensitivity |
| 30 | hierarchical shrinkage、WAIC boundary | tenor別5日変化 | exchangeability failure |
| 31 | DAG、mixture、HMM、EM | NS factor-change states | label/state truth claim |
| 32 | MH、ESS、split-(\hat R)、approximation boundary | predictive uncertainty | trace-only diagnosis |

CoreはNumPy/SciPyによる透明な共役計算、random-walk MH、Gaussian HMM。HMC/NUTS/VI/SMCは理論・診断のAdvanced範囲で、finite differenceをautomatic differentiationと呼ばない。

In [4]:
horizon = 5
origins = np.arange(curve_yields.shape[0] - horizon)
target_changes_bp = (curve_yields[origins + horizon] - curve_yields[origins]) * 100.0
target_dates = curve_dates[origins + horizon]
training_targets = target_changes_bp[target_dates <= train_end_date]

prior_rng = task_rng(1)
prior_mean_draws = prior_rng.normal(loc=0.0, scale=5.0, size=4000)
prior_predictive = prior_mean_draws + prior_rng.normal(
    scale=training_targets[:, 3].std(ddof=1), size=4000
)
fig = go.Figure()
fig.add_histogram(x=training_targets[:, 3], name="observed training targets", histnorm="probability density", opacity=0.6)
fig.add_histogram(x=prior_predictive, name="prior predictive", histnorm="probability density", opacity=0.6)
fig.update_layout(
    title="Prior predictive scale check for five-publication 10y changes",
    xaxis_title="Change (bp)",
    barmode="overlay",
    template="plotly_white",
)
fig.show()

## 2. Uncertainty contract

- Bayesian linear modelはparameter uncertaintyとobservation noiseを積分したposterior predictiveを返す。
- HMM EMはpoint-estimated parameterの下のpredictive distributionで、full Bayesian posterior predictiveではない。
- coverage、interval width、log predictive density、point RMSEを別々に報告する。
- HMM state probabilityはmodel内の条件付き確率で、外部の観測済みmarket regime labelではない。

## 3. 失敗モード

- priorを隠して「dataだけ」の結論と呼ぶ
- posterior intervalとfrequentist repeated-sampling CIを同義にする
- testを見てstate数やprior scaleを選ぶ
- EMのstate確率をparameter posterior uncertaintyと呼ぶ
- trace plotだけでMCMC convergenceを宣言する

## 4. 段階別演習

### 基礎

1. prior predictiveとposterior predictiveの条件付け集合を書け。
2. 5日先targetのtraining/validation/test件数を数えよ。

### 標準

3. prior standard deviation 1/5/20 bpの感応度を比較せよ。
4. coverageとwidthを同時に採用するgateを書け。

### 研究

5. full Bayesian switching state-space modelへ進む前のsimulation-based calibration計画を書け。

## 5. Exit Criteria

- [ ] B7と同じdata・target・outer testを使う
- [ ] posterior predictiveとHMM conditional predictiveを区別した
- [ ] point errorとcoverage/log scoreを分離した
- [ ] state labelを観測真値と呼ばない
- [ ] prior sensitivityとMCMC diagnosticsを必須にした

## 6. 出典


- [Gelman et al., Bayesian Data Analysis, 3rd ed.](https://sites.stat.columbia.edu/gelman/book/)
- [Gelman et al., Bayesian Workflow](https://arxiv.org/abs/2011.01808)
- [Vehtari, Gelman, and Gabry, Practical Bayesian model evaluation](https://doi.org/10.1007/s11222-016-9696-4)

- [Rabiner (1989), A Tutorial on Hidden Markov Models](https://www.cs.cmu.edu/~durand/03-711/Readings/Rabiner89.pdf)
- [Vehtari et al., Rank-normalization, folding, and localization](https://arxiv.org/abs/1903.08008)
- [Stan Reference Manual — MCMC Sampling](https://mc-stan.org/docs/reference-manual/mcmc.html)